# Thermal explosion

In [ ]:
from scipy.sparse import diags, block_diag, eye
from scipy.sparse.linalg import spsolve, splu
from scipy.integrate import solve_ivp

import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Model with fuel consumption neglected

$$
\partial_\tau \theta - {\displaystyle 1 \over \displaystyle \lambda} \partial_{\xi\xi} \theta = \exp(\theta)
$$

In [ ]:
class thermal_explosion:
    def __init__(self, lamb, xmin, xmax, nx):
        self.lamb = lamb
        self.xmin = xmin
        self.xmax = xmax
        self.nx = nx
        self.dx = (xmax-xmin)/(nx+1)
        
        oneoverlambdadxdx = 1./(lamb*(self.dx**2))
        diag0 = np.repeat(-2., nx)
        diag1 = np.repeat(1., nx-1)
        self.a = diags([diag1, diag0, diag1], [-1, 0, 1], format="csc")
        self.a = oneoverlambdadxdx*self.a

    def fr(self, theta):
        return np.exp(theta)

### The function $\Psi$

In [ ]:
x = np.linspace(0,20,1000)
# Psi = np.exp(-x/2) * np.arccosh(np.exp(x/2))
Psi = 1/2 * np.exp(-x/2) * np.log( (1+np.sqrt(1-np.exp(-x))) / (1-np.sqrt(1-np.exp(-x))) )
Psi_max = np.max(Psi)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=Psi))
fig.update_layout(xaxis_title="x", yaxis_title="Psi", title=f"Maximal value of Psi: {Psi_max:.4f}")
fig.show()

### Integration with IMEX scheme

In [ ]:
def imex(tini, tend, nt, yini, a, f, nt_eval=None, param=None):
    
    if param is not None:
          f = lambda x, f=f : f(x, *param)

    dt = (tend-tini) / (nt-1)
    
    lu = splu(eye(yini.size, format="csc") - dt*a)

    if nt_eval is None: nt_eval = (nt,)
        
    y = np.zeros((len(nt_eval), yini.size))
        
    yn = np.copy(yini)

    it_eval = 0 
    for it in range(1,nt):
        yn = lu.solve(yn + dt*f(yn))
        if (it+1 in nt_eval):
            y[it_eval] = yn 
            it_eval = it_eval+1
            
    if nt_eval is None: y[-1] = yn

    return y

In [ ]:
## model parameters
xmin = 0
xmax = 2
lamb = 0.87

## time integration
tini = 0
tend = 10
nt = 10001
dt = (tend-tini) / (nt-1)

# nb interior points 
nx = 99

# model
tem = thermal_explosion(lamb, xmin, xmax, nx)

# diffusion matrix
a  = tem.a

# rhs 
fr = tem.fr

# integration
theta_ini = np.zeros(nx)
theta = imex(tini, tend, nt, theta_ini, a, fr, nt_eval=(nt//4, nt//2, (3*nt)//4, nt))

# output
xi = np.linspace(xmin, xmax, nx+2)
fig = go.Figure()
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta_ini,[0])), name=f't = {tini:.2f}'))
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[0],[0])),  name=f't = {tend/4:.2f}'))
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[1],[0])),  name=f't = {tend/2:.2f}'))
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[2],[0])),  name=f't = {3*tend/4:.2f}'))
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[-1],[0])), name=f't = {tend:.2f}'))

fig.update_layout(xaxis_title="xi", yaxis_title="theta")
fig.show()

## Model with fuel consumption

$$
\left\{\begin{aligned}
\partial_\tau \theta - {\displaystyle 1 \over \displaystyle \lambda} \partial_{\xi\xi} \theta & = \exp(\theta)\, Y\\
\partial_\tau Y - {\displaystyle 1 \over \displaystyle \lambda} \partial_{\xi\xi} Y & = - \epsilon \exp(\theta)\, Y
\end{aligned}\right.
$$

In [ ]:
class thermal_explosion_fuel:

    def __init__(self, lamb, eps, xmin, xmax, nx):
        self.lamb = lamb
        self.eps  = eps
        self.xmin = xmin
        self.xmax = xmax
        self.nx = nx
        self.dx = (xmax-xmin)/(nx-1)
        self.oneoverlambdadxdx = 1./(lamb*(self.dx**2))     
        
        # laplacian matrix with homegenous Dirichlet conditions
        diag0 = np.repeat(-2., nx-2)
        diag1 = np.repeat(1., nx-3)
        lapd = diags([diag1, diag0, diag1], [-1, 0, 1], format="csc")
        lapd = self.oneoverlambdadxdx*lapd
        # laplacian matrix with homegenous Neumman conditions (order 2)
        diag3 = np.repeat(-2., nx)
        diag4 = np.repeat([2., 1., 2], [1, nx-2, 1])
        lapn = diags([diag4[1:], diag3, diag4[:-1]], [-1, 0, 1])
        lapn = self.oneoverlambdadxdx*lapn
        self.a = laplap = block_diag((lapd, lapn))

    def fr(self, y):
        nx = self.nx
        theta, Y = np.split(y, [nx-2])
        tmp = np.exp(theta)*Y[1:-1]
        return np.concatenate((tmp, -eps*np.concatenate(([Y[0]], tmp, [Y[-1]]))))

    def fcn(self, t, y):
        
        nx = self.nx
        oneoverlambdadxdx = self.oneoverlambdadxdx
        
        theta, Y = np.split(y, [nx-2])

        dtheta = np.empty(nx-2)
        dY     = np.empty(nx)
            
        dtheta[0]    = oneoverlambdadxdx * (-2*theta[0] + theta[1]) + np.exp(theta[0]) * Y[1]      
        dtheta[1:-1] = oneoverlambdadxdx * (theta[:-2] - 2*theta[1:-1] + theta[2:]) + np.exp(theta[1:-1]) * Y[2:-2]
        dtheta[-1]   = oneoverlambdadxdx * (theta[-2] - 2*theta[-1]) + np.exp(theta[-1]) * Y[-2]   
        
        dY[0]    = 2*oneoverlambdadxdx * (-Y[0] + Y[1]) - eps * np.exp(0.) * Y[0]       
        dY[1:-1] = oneoverlambdadxdx * (Y[:-2] - 2*Y[1:-1] + Y[2:]) - eps * np.exp(theta[:]) * Y[1:-1]
        dY[-1]   = 2*oneoverlambdadxdx * (Y[nx-2] - Y[nx-1]) - eps * np.exp(0.) * Y[-1] 
        
        return np.hstack((dtheta, dY))

### Integration with IMEX scheme

In [ ]:
## model parameters
xmin = 0
xmax = 2
lamb = 0.88
eps = 0.002

## time integration
tini = 0
tend = 10
nt = 10001
dt = (tend-tini) / (nt-1)

# nb space points including boundaries 
nxib = 101 
xi = np.linspace(xmin, xmax, nxib)

# model
temf = thermal_explosion_fuel(lamb, eps, xmin, xmax, nxib)

# diffusion matrix
a  = temf.a

# rhs 
fr = temf.fr

# init sol
theta_ini = np.zeros(nxib-2)
Y_ini     = np.ones(nxib)
#Y_ini     = np.ones(nxib) + 0.5*(xi-1)
sol_ini   = np.hstack((theta_ini, Y_ini))

# IMEX integration
sol = imex(tini, tend, nt, sol_ini, a, fr, nt_eval=(nt//4, nt//2, (3*nt)//4, nt))
theta = sol[:,:nxib-2]
Y = sol[:,nxib-2:]

# output
fig = make_subplots(rows=2, cols=1)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],sol_ini[:nxib-2],[0])), name=f't = {tini:.2f}', line_color='rgb(76,114,176)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=sol_ini[nxib-2:], showlegend=False, line_color='rgb(76,114,176)'), col=1, row=2)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[0],[0])), name=f't = {tend/4:.2f}', line_color='rgb(221,132,82)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y[0], showlegend=False, line_color='rgb(221,132,82)' ), col=1, row=2)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[1],[0])), name=f't = {tend/2:.2f}', line_color='rgb(85,168,104)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y[1], showlegend=False, line_color='rgb(85,168,104)'), col=1, row=2)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[2],[0])), name=f't = {3*tend/4:.2f}', line_color='rgb(196,78,82)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y[2], showlegend=False, line_color='rgb(196,78,82)'), col=1, row=2)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[-1],[0])), name=f't = {tend:.2f}', line_color='rgb(129,114,179)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y[-1], showlegend=False, line_color='rgb(129,114,179)'), col=1, row=2)
fig.update_xaxes(title_text="x", col=1)
fig.update_yaxes(title_text="theta", row=1)
fig.update_yaxes(title_text="Y", row=2)
fig.update_layout(height=1000)
fig.show()

### Integration with RADAUIIA

In [ ]:
## model parameters
xmin = 0
xmax = 2
lamb = 0.88
eps = 0.002

## time integration
tini = 0
tend = 10
tspan = (tini, tend)

# nb space points including boundaries 
nxib = 101 
xi = np.linspace(xmin, xmax, nxib)

# model
temf = thermal_explosion_fuel(lamb, eps, xmin, xmax, nxib)

fcn  = temf.fcn

# init sol
theta_ini = np.zeros(nxib-2)
Y_ini     = np.ones(nxib)
sol_ini   = np.hstack((theta_ini, Y_ini))

## RADAUIIA
tol = 1e-10
sol_rad = solve_ivp(fcn, (tini, tend), sol_ini, method="Radau", dense_output=True, rtol=tol, atol=tol)
print("RadauIIA integration :")
print(f"  Number of accepted steps               = {sol_rad.t.size-1}")
print(f"  Number of function evaluations         = {sol_rad.nfev}")
print(f"  Number of evaluations of the Jacobian  = {sol_rad.njev}")
print(f"  Number of LU decompositions            = {sol_rad.nlu}")
print(f"  Final time reached                     = {sol_rad.t[-1]}")

dt_rad = np.array(sol_rad.t[1:]) - np.array(sol_rad.t[:-1])
t_rad  = np.array(sol_rad.t[:-1]) + 0.5*dt_rad

# output
tend = sol_rad.t[-1]
t_eval=(tend/4, tend/2, (3*tend)/4, tend)

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1, subplot_titles=("Theta", "Y"))
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta_ini,[0])), name=f't = {tini:.2f}', line_color='rgb(76,114,176)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y_ini, showlegend=False, line_color='rgb(76,114,176)'), col=1, row=2)
lc = ['rgb(221,132,82)', 'rgb(85,168,104)', 'rgb(196,78,82)', 'rgb(129,114,179)']
for i, ti in enumerate(t_eval):
    sol = sol_rad.sol(ti) 
    theta, Y = np.split(sol, [nxib-2])
    fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta,[0])), name=f't = {ti:.2f}', line_color=lc[i]), row=1, col=1)
    fig.add_trace(go.Scatter(x=xi, y=Y, showlegend=False, line_color=lc[i]), row=2, col=1)
fig.update_layout(height=1000)
fig.show()

fig_dt = go.Figure()
fig_dt.add_trace(go.Bar(x=t_rad, y=dt_rad, width=dt_rad, showlegend=False))
fig_dt.update_xaxes(title_text="t")
fig_dt.update_yaxes(title_text="dt")
fig_dt.show()


## Model with fuel consumption and without linearization

$$
\left\{\begin{aligned}
\partial_\tau \theta - {\displaystyle 1 \over \displaystyle \lambda} \partial_{\xi\xi} \theta & = 
\exp\Big(\frac{\theta}{1+\theta/\beta}\Big)\, Y\\
\partial_\tau Y - {\displaystyle 1 \over \displaystyle \lambda} \partial_{\xi\xi} Y & = - \epsilon \exp\Big(\frac{\theta}{1+\theta/\beta}\Big)\, Y
\end{aligned}\right.
$$

In [ ]:
class thermal_explosion_fuel_nl:

    def __init__(self, lamb, beta, eps, xmin, xmax, nx):
        self.lamb = lamb
        self.beta = beta
        self.eps  = eps
        self.xmin = xmin
        self.xmax = xmax
        self.nx = nx
        self.dx = (xmax-xmin)/(nx-1)
        self.oneoverlambdadxdx = 1./(lamb*(self.dx**2))     
        
        # laplacian matrix with homegenous Dirichlet conditions
        diag0 = np.repeat(-2., nx-2)
        diag1 = np.repeat(1., nx-3)
        lapd = diags([diag1, diag0, diag1], [-1, 0, 1], format="csc")
        lapd = self.oneoverlambdadxdx*lapd
        # laplacian matrix with homegenous Neumman conditions (order 2)
        diag3 = np.repeat(-2., nx)
        diag4 = np.repeat([2., 1., 2], [1, nx-2, 1])
        lapn = diags([diag4[1:], diag3, diag4[:-1]], [-1, 0, 1])
        lapn = self.oneoverlambdadxdx*lapn
        self.a = laplap = block_diag((lapd, lapn))

    def fr(self, y):
        nx = self.nx
        theta, Y = np.split(y, [nx-2])
        tmp = np.exp(theta/(1+theta/beta))*Y[1:-1]
        return np.concatenate((tmp, -eps*np.concatenate(([Y[0]], tmp, [Y[-1]]))))

    def fcn(self, t, y):
        
        nx = self.nx
        oneoverlambdadxdx = self.oneoverlambdadxdx
        
        theta, Y = np.split(y, [nx-2])

        dtheta = np.empty(nx-2)
        dY     = np.empty(nx)

        dtheta[0]    = oneoverlambdadxdx * (-2*theta[0] + theta[1]) + np.exp(theta[0]/(1.+theta[0]/beta)) * Y[1]      
        dtheta[1:-1] = oneoverlambdadxdx * (theta[:-2] - 2*theta[1:-1] + theta[2:]) + np.exp(theta[1:-1]/(1.+theta[1:-1]/beta)) * Y[2:-2]
        dtheta[-1]   = oneoverlambdadxdx * (theta[-2] - 2*theta[-1]) + np.exp(theta[-1]/(1.+theta[-1]/beta)) * Y[-2]   
        
        dY[0]    = 2*oneoverlambdadxdx * (-Y[0] + Y[1]) - eps * np.exp(0./(1.+theta[0]/beta)) * Y[0]       
        dY[1:-1] = oneoverlambdadxdx * (Y[:-2] - 2*Y[1:-1] + Y[2:]) - eps * np.exp(theta[:]/(1.+theta[:]/beta)) * Y[1:-1]
        dY[-1]   = 2*oneoverlambdadxdx * (Y[nx-2] - Y[nx-1]) - eps * np.exp(0./(1.+theta[-1]/beta)) * Y[-1] 
                    
        return np.hstack((dtheta, dY))

### Integration with IMEX scheme

In [ ]:
## model parameters
xmin = 0
xmax = 2
lamb = 0.95
eps = 0.002
beta = 30

## time integration
tini = 0
tend = 10.8
# tend = 10.81
nt = 100001
dt = (tend-tini) / (nt-1)

# nb space points including boundaries 
nxib = 101 
xi = np.linspace(xmin, xmax, nxib)

# model
temfnl = thermal_explosion_fuel_nl(lamb, beta, eps, xmin, xmax, nxib)

# diffusion matrix
a  = temfnl.a

# rhs 
fr = temfnl.fr

# init sol
theta_ini = np.zeros(nxib-2)
Y_ini     = np.ones(nxib)
#Y_ini     = np.ones(nxib) + 0.5*(xi-1)
sol_ini   = np.hstack((theta_ini, Y_ini))

# IMEX integration
sol = imex(tini, tend, nt, sol_ini, a, fr, nt_eval=(nt//4, nt//2, (3*nt)//4, nt))
theta = sol[:,:nxib-2]
Y = sol[:,nxib-2:]

# output
fig = make_subplots(rows=2, cols=1)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],sol_ini[:nxib-2],[0])), name=f't = {tini:.3f}', line_color='rgb(76,114,176)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=sol_ini[nxib-2:], showlegend=False, line_color='rgb(76,114,176)'), col=1, row=2)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[0],[0])), name=f't = {tend/4:.3f}', line_color='rgb(221,132,82)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y[0], showlegend=False, line_color='rgb(221,132,82)' ), col=1, row=2)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[1],[0])), name=f't = {tend/2:.3f}', line_color='rgb(85,168,104)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y[1], showlegend=False, line_color='rgb(85,168,104)'), col=1, row=2)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[2],[0])), name=f't = {3*tend/4:.3f}', line_color='rgb(196,78,82)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y[2], showlegend=False, line_color='rgb(196,78,82)'), col=1, row=2)
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta[-1],[0])), name=f't = {tend:.3f}', line_color='rgb(129,114,179)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y[-1], showlegend=False, line_color='rgb(129,114,179)'), col=1, row=2)
fig.update_xaxes(title_text="x", col=1)
fig.update_yaxes(title_text="theta", row=1)
fig.update_yaxes(title_text="Y", row=2)
fig.update_layout(height=1000)
fig.show()

### Integration with RADAUIIA

In [ ]:
## model parameters
xmin = 0
xmax = 2
lamb = 0.95
eps = 0.002
beta = 30

## time integration
tini = 0
tend = 10.7928
tspan = (tini, tend)

# nb space points including boundaries 
nxib = 101 
xi = np.linspace(xmin, xmax, nxib)

# model
temfnl = thermal_explosion_fuel_nl(lamb, beta, eps, xmin, xmax, nxib)

fcn = temfnl.fcn

# init sol
theta_ini = np.zeros(nxib-2)
Y_ini     = np.ones(nxib)
sol_ini   = np.hstack((theta_ini, Y_ini))

## RADAUIIA
tol = 1e-6
sol_rad = solve_ivp(fcn, (tini, tend), sol_ini, method="Radau", dense_output=True, rtol=tol, atol=tol)
print("RadauIIA integration :")
print(f"  Number of accepted steps               = {sol_rad.t.size-1}")
print(f"  Number of function evaluations         = {sol_rad.nfev}")
print(f"  Number of evaluations of the Jacobian  = {sol_rad.njev}")
print(f"  Number of LU decompositions            = {sol_rad.nlu}")
print(f"  Final time reached                     = {sol_rad.t[-1]}")

dt_rad = np.array(sol_rad.t[1:]) - np.array(sol_rad.t[:-1])
t_rad  = np.array(sol_rad.t[:-1]) + 0.5*dt_rad

# output
t_eval=(tend/4, tend/2, (3*tend)/4, tend)
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1, subplot_titles=("Theta", "Y"))
fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta_ini,[0])), name=f't = {tini:.3f}', line_color='rgb(76,114,176)'), col=1, row=1)
fig.add_trace(go.Scatter(x=xi, y=Y_ini, showlegend=False, line_color='rgb(76,114,176)'), col=1, row=2)
lc = ['rgb(221,132,82)', 'rgb(85,168,104)', 'rgb(196,78,82)', 'rgb(129,114,179)']
for i, ti in enumerate(t_eval):
    sol = sol_rad.sol(ti) 
    theta, Y = np.split(sol, [nxib-2])
    fig.add_trace(go.Scatter(x=xi, y=np.concatenate(([0],theta,[0])), name=f't = {ti:.3f}', line_color=lc[i]), row=1, col=1)
    fig.add_trace(go.Scatter(x=xi, y=Y, showlegend=False, line_color=lc[i]), row=2, col=1)
fig.update_layout(height=1000)
fig.show()

fig_dt = go.Figure()
fig_dt.add_trace(go.Bar(x=t_rad, y=dt_rad, width=dt_rad, showlegend=False))
fig_dt.update_xaxes(title_text="t")
fig_dt.update_yaxes(title_text="dt")
fig_dt.show()